Dependencies

In [14]:
import pandas as pd
import chromadb
import nltk
import os
import json
from nltk.corpus import stopwords
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.vectorizers import ClassTfidfTransformer
from bertopic.representation import MaximalMarginalRelevance, KeyBERTInspired
from bertopic import BERTopic
from dotenv import load_dotenv

In [15]:
load_dotenv("../.env")

# Credentials
hf_token = os.getenv("HF_TOKEN")
# print(hf_token)

# Path
topic_data_path = os.getenv("FILE_TOPIC_DATA_PATH")
file_clustering_data_path = os.getenv("FILE_CLUSTERING_DATA_PATH")
db_path = os.getenv("DB_PATH")
# print(file_clustering_data_path)

# Name
db_name = os.getenv("DB_NAME")
model_embedding_name = os.getenv("MODEL_EMBEDDING_NAME")

# Paksa Pandas untuk tidak menyembunyikan baris (Tampilkan SEMUA)
pd.set_option('display.max_rows', None) 
# Paksa Pandas menampilkan teks kolom secara utuh (tidak dipotong titik-titik "...")
pd.set_option('display.max_colwidth', None)

Getting Data

In [16]:
# Pengambilan Data
print("📦 Menarik Vektor dari Database Lokal...")
client = chromadb.PersistentClient(path=db_path)
collection = client.get_collection(name=db_name)
print("Total data di ChromaDB:", collection.count())
data_db = collection.get(include=['embeddings', 'documents', 'metadatas'])

docs = data_db['documents']
embedding_vectors = data_db['embeddings']
metadata = data_db['metadatas']

# Persiapan Stopwords
nltk.download('stopwords', quiet=True)
stopwords_list = stopwords.words('indonesian')
custom_stopwords = ['sarah', 'seperti', 'bisa', 'menjadi', 'karena', 'untuk', 'dengan', 'itu', 'ini', 'yang', 'dan', 'di', 'dalam']
stopwords_list.extend(custom_stopwords)

📦 Menarik Vektor dari Database Lokal...
Total data di ChromaDB: 241


Embedding Model

In [17]:
embedding_model = SentenceTransformer(
    model_name_or_path=model_embedding_name,
    token=hf_token
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3822.24it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Dimensional Reduction Model

In [18]:
umap_model = UMAP(
    n_neighbors=10,
    n_components=5,
    min_dist=0.0,
    metric='cosine',
    random_state=42
)

Clustering Model

In [19]:
hdbscan_model = HDBSCAN(
    min_cluster_size=4,
    min_samples=2,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

Vectorizer Model

In [20]:
vectorizer_model = CountVectorizer(
    stop_words=stopwords_list,
    ngram_range=(1, 6),
    # min_df=5
)

c-TF-IDF Model

In [21]:
ctfidf_model = ClassTfidfTransformer(
    bm25_weighting=True,
    reduce_frequent_words=True
)

Fine-tuning / Representation Model

In [22]:
# Multi-representation model implementation
keybert_model = KeyBERTInspired()
mmr_model = MaximalMarginalRelevance(diversity=0.3)

representation_models = {
    "KeyBERT":keybert_model,
    "MMR":mmr_model,
    "Main":keybert_model
}

BERTopic Training

In [23]:
print("⚙️ Memulai proses ekstraksi dan clustering topik (Kecepatan Tinggi)...")

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    representation_model=representation_models,
    verbose=True
)

topic, probabilities = topic_model.fit_transform(docs, embeddings=embedding_vectors)

print("\n" + "="*50)
print("🏆 HASIL CLUSTERING BERTOPIC")
print("="*50)

# Memaksa 40 topik yang terbentuk untuk menyusut dan bergabung menjadi 10 Topik Utama saja!
# topic_model.reduce_topics(docs, nr_topics=10)

df_topik_info = topic_model.get_topic_info()

# Sekarang jalankan print ini tanpa .head(10)
print(df_topik_info[['Topic', 'Count', 'Name']])

2026-04-30 10:30:07,654 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


⚙️ Memulai proses ekstraksi dan clustering topik (Kecepatan Tinggi)...


2026-04-30 10:30:08,100 - BERTopic - Dimensionality - Completed ✓
2026-04-30 10:30:08,102 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-30 10:30:08,119 - BERTopic - Cluster - Completed ✓
2026-04-30 10:30:08,125 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-30 10:30:37,669 - BERTopic - Representation - Completed ✓



🏆 HASIL CLUSTERING BERTOPIC
    Topic  Count  \
0      -1     12   
1       0     31   
2       1     14   
3       2     14   
4       3     13   
5       4     12   
6       5     12   
7       6     11   
8       7     11   
9       8     10   
10      9      8   
11     10      8   
12     11      7   
13     12      7   
14     13      7   
15     14      7   
16     15      6   
17     16      6   
18     17      6   
19     18      5   
20     19      5   
21     20      5   
22     21      4   
23     22      4   
24     23      4   
25     24      4   
26     25      4   
27     26      4   

                                                                                                                                                     Name  
0                                                                        -1_perkembangan teknologi_inovasi_berbasis masyarakat_sampah berbasis masyarakat  
1                                                                             

In [24]:
cluster = 0
raw_topic = topic_model.get_topic(cluster)

print("🌟 DAFTAR KATA KUNCI BERSIH:")
for kata, skor in raw_topic:
    # Ubah np.float32 menjadi float biasa bawaan Python, lalu bulatkan 4 angka di belakang koma
    skor_bersih = float(skor) 
    print(f"- {kata} (Skor: {skor_bersih:.4f})")

🌟 DAFTAR KATA KUNCI BERSIH:
- telkom indonesia (Skor: 0.5522)
- indonesia (Skor: 0.5012)
- infrastruktur (Skor: 0.4676)
- transformasi digital (Skor: 0.4479)
- industri telekomunikasi (Skor: 0.4459)
- infrastructure (Skor: 0.4395)
- jaringan (Skor: 0.4348)
- digital transformation (Skor: 0.4213)
- industri (Skor: 0.4077)
- teknologi (Skor: 0.4040)


In [25]:
# 1. Buat DataFrame dari metadata (judul, slug, dll) yang kita tarik dari ChromaDB
df_raw = pd.DataFrame(metadata)

# 2. Tempelkan teks asli agar bisa dibaca
df_raw['article_text'] = docs

# 3. INI DIA FUNGSI UTAMA VARIABEL 'topics'!
# Tempelkan list topics sebagai kolom baru di tabelmu
df_raw['id_topic'] = topic

# 4. Ambil nama topik dari BERTopic dan petakan ke tabel
mapping_topic_name = topic_model.get_topic_info().set_index('Topic')['Name'].to_dict()
df_raw['name_topic'] = df_raw['id_topic'].map(mapping_topic_name)

# 5. Masukkan hasil probabilitas dari BERTopic ke dalam tabel
df_raw['skor_cf'] = probabilities

# 6. Filtering (Aturan > 50%)
min_cf_range = 0.50

# Buang Outlier bawaan (-1) DAN buang artikel yang keyakinannya di bawah 50%
df_final = df_raw[
    (df_raw['id_topic'] != -1) & 
    (df_raw['skor_cf'] >= min_cf_range)
]

print(f"Total Artikel Awal: {len(df_raw)}")
print(f"Total Artikel Setelah Filtering (>{min_cf_range*100}% Yakin): {len(df_final)}")

# 3. Analisis Ulang (Mencari Niche dari data yang sudah sangat murni)
final_distribution = df_final['name_topic'].value_counts().reset_index()
final_distribution.columns = ['name_topic', 'article_count']
full_distribution = final_distribution.sort_values(by='article_count', ascending=True)
niche_topic = full_distribution.head(1)

print("\n" + "="*50)
print("🎯 REKOMENDASI TOPIK NICHE (PALING SEDIKIT DIBEDAH)")
print("="*50)
print(f"Topik: {niche_topic.iloc[0]['name_topic']}")
print(f"Jumlah: {niche_topic.iloc[0]['article_count']} Artikel")
print("Catatan: Topik ini memiliki potensi tinggi untuk diangkat menjadi bahan artikel baru karena belum banyak yang membahasnya di platform!")

# Urutan lengkap
print("\n📋 Daftar Urutan Topik (Terkecil ke Terbesar):")
print(full_distribution.to_string(index=False))

# print("\n📊 Distribusi Topik Setelah Filtering:")
# print(final_distribution)

# Simpan ke CSV untuk diekspor!
df_final.to_csv(file_clustering_data_path, index=False)
print("💾 Data berhasil diekspor ke CSV lengkap dengan label topiknya!")

Total Artikel Awal: 241
Total Artikel Setelah Filtering (>50.0% Yakin): 214

🎯 REKOMENDASI TOPIK NICHE (PALING SEDIKIT DIBEDAH)
Topik: 24_tools praktis ui designer_chatbot_assistant otomatis_tools praktis ui
Jumlah: 4 Artikel
Catatan: Topik ini memiliki potensi tinggi untuk diangkat menjadi bahan artikel baru karena belum banyak yang membahasnya di platform!

📋 Daftar Urutan Topik (Terkecil ke Terbesar):
                                                                                                                                           name_topic  article_count
                                                                             24_tools praktis ui designer_chatbot_assistant otomatis_tools praktis ui              4
                                                    21_transformasi digital rumah sakit_kesehatan digital_digital rumah sakit_sistem kesehatan modern              4
                               26_bisnis berbasis sustainability_berbasis sustainability_dipelaja

In [26]:
def ektract_info_cluster(raw_text):
    sliced_text = raw_text.split('_')
    cluster_id = int(sliced_text[0])
    keywords = [k.strip() for k in sliced_text[1:]]
    main_cluster_name = keywords[0].title() 
    return cluster_id, main_cluster_name, keywords

# PROSES PEMBUATAN DATA JSON

# 1. Tentukan target niche (misal: 3 cluster terkecil)
niche_targets = 3
niche_table = full_distribution.head(niche_targets)

# 2. Kumpulkan ID Cluster yang berstatus "Niche/Rekomendasi"
niche_cluster_ids = []
for text in niche_table['name_topic']:
    cid, _, _ = ektract_info_cluster(text)
    niche_cluster_ids.append(cid)

# 3. Ekstrak Struktur Cluster Lengkap beserta Flag Rekomendasinya
final_cluster_list = []

for text in full_distribution['name_topic']:
    cid, cname, ckeys = ektract_info_cluster(text)
    
    # Jika ID ini ada di dalam daftar niche_cluster_ids, maka True. Jika tidak, False.
    recommend_status = cid in niche_cluster_ids
    
    final_cluster_list.append({
        "cluster_id": cid,
        "cluster_name": cname,
        "is_recommended": recommend_status,  
        "keywords": ckeys
    })

# Urutkan dari index pertama ke n
final_cluster_list = sorted(final_cluster_list, key=lambda x: x['cluster_id'])

# 4. Susun Format Final
output_json = {
    "metadata": {
        "total_clusters": len(final_cluster_list),
        "total_recommended": len(niche_cluster_ids)
    },
    "clusters": final_cluster_list
}

# 5. Simpan ke File
with open(topic_data_path, "w", encoding="utf-8") as f:
    json.dump(output_json, f, ensure_ascii=False, indent=4)

print("✅ JSON Terstruktur dengan Flag Rekomendasi Berhasil Dibuat!")

✅ JSON Terstruktur dengan Flag Rekomendasi Berhasil Dibuat!
